# Build LaTeX PDF locally

Patches `myst.yml` to use the LaTeX export config, builds the PDF into `exports/`, then restores `myst.yml`.

**Prerequisites (run once as admin):**
```bash
sudo apt-get install -y texlive-latex-base texlive-latex-recommended texlive-fonts-recommended \
  texlive-xetex texlive-latex-extra latexmk fonts-lmodern nodejs npm
pip install mystmd
```

In [ ]:
import yaml, copy, pathlib

myst_path = pathlib.Path("..") / "myst.yml"
config = yaml.safe_load(myst_path.read_text())
original_extends = copy.deepcopy(config.get("extends", []))

config["extends"] = ["toc.yml", "export latex.yml", "plugins.yml"]
myst_path.write_text(yaml.dump(config, allow_unicode=True, sort_keys=False))
print("myst.yml patched for LaTeX build")

In [ ]:
import subprocess

try:
    subprocess.run(["myst", "build", "--pdf"], cwd=myst_path.parent, check=True)
finally:
    config["extends"] = original_extends
    myst_path.write_text(yaml.dump(config, allow_unicode=True, sort_keys=False))
    print("myst.yml restored")

In [ ]:
exports = sorted((myst_path.parent / "exports").glob("*.pdf"))
if exports:
    for p in exports:
        print(p.name, f"({p.stat().st_size / 1024:.1f} KB)")
else:
    print("No PDFs found in exports/")